# Premier League — App de Predicciones

**Requisito:** Tener `modelo_premier.pkl` generado por `premier_league_train.ipynb`

```
Flujo de uso:
─────────────────────────────────────────────────────
Antes de la jornada  →  Celdas 1, 2, 3, 4, 5
                        Predecir partidos y registrar

Después de cada partido →  Celda 6
                           Registrar resultado real

Ver resumen             →  Celda 7
─────────────────────────────────────────────────────
```

In [29]:
# ─────────────────────────────────────────────
# CELDA 1 — Imports y carga del modelo
# Correr siempre al abrir el notebook
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import joblib
# import pickle

import json

# main.py



# # Cargar modelo desde disco
# with open('modelo_premier.pkl', 'rb') as f:
#     datos = pickle.load(f)

# model_v3        = datos['model_v3']
# feature_cols_v3 = datos['feature_cols_v3']
# le              = datos['le']
# df              = datos['df']


# Cargar modelo desde disco
with open('modelo_premier.pkl', 'rb') as f:
    datos = joblib.load('modelo_premier.pkl')

model_v3 = datos['model_v3']
feature_cols_v3 = datos['feature_cols_v3']
le              = datos['le']
df              = datos['df']

print('Modelo cargado correctamente')
print(f'Features: {len(feature_cols_v3)}')
print(f'Partidos históricos: {len(df)}')
print(f'Equipos disponibles:')
print(sorted(df['HomeTeam'].unique()))

Modelo cargado correctamente
Features: 52
Partidos históricos: 2209
Equipos disponibles:
['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Ipswich', 'Leeds', 'Leicester', 'Liverpool', 'Luton', 'Man City', 'Man United', 'Newcastle', 'Norwich', "Nott'm Forest", 'Sheffield United', 'Southampton', 'Sunderland', 'Tottenham', 'Watford', 'West Brom', 'West Ham', 'Wolves']


In [30]:
# ─────────────────────────────────────────────
# CELDA 2 — Funciones del modelo
# Correr siempre al abrir el notebook
# ─────────────────────────────────────────────

def get_team_stats_v2(team, date, df, n_recent=5):
    """Stats combinadas: forma reciente + temporada actual."""
    home_all = df[(df['HomeTeam'] == team) & (df['Date'] < date)]
    away_all = df[(df['AwayTeam'] == team) & (df['Date'] < date)]
    all_games = pd.concat([home_all, away_all]).sort_values('Date')

    if len(all_games) < 3:
        return None

    current_season = df[df['Date'] < date]['season'].iloc[-1]
    home_szn  = home_all[home_all['season'] == current_season]
    away_szn  = away_all[away_all['season'] == current_season]
    season_games = pd.concat([home_szn, away_szn]).sort_values('Date')
    recent = all_games.tail(n_recent)

    def calc_stats(games, team):
        if len(games) == 0:
            return None
        gf=gc=sot_f=sot_c=wins=draws=home_wins=home_games=0
        for _, r in games.iterrows():
            is_home = r['HomeTeam'] == team
            gf    += r['FTHG'] if is_home else r['FTAG']
            gc    += r['FTAG'] if is_home else r['FTHG']
            sot_f += r['HST']  if is_home else r['AST']
            sot_c += r['AST']  if is_home else r['HST']
            won  = (is_home and r['FTR']=='H') or (not is_home and r['FTR']=='A')
            drew = r['FTR'] == 'D'
            if won:  wins  += 1
            if drew: draws += 1
            if is_home:
                home_games += 1
                if r['FTR'] == 'H': home_wins += 1
        m = len(games)
        return {
            'gf_pg':     gf/m,
            'gc_pg':     gc/m,
            'dif_goles': (gf-gc)/m,
            'sot_f_pg':  sot_f/m,
            'sot_c_pg':  sot_c/m,
            'win_rate':  wins/m,
            'draw_rate': draws/m,
            'home_wr':   home_wins/home_games if home_games > 0 else wins/m,
        }

    recent_s = calc_stats(recent, team)
    season_s = calc_stats(season_games, team) if len(season_games) >= 3 else recent_s

    if recent_s is None:
        return None

    combined = {}
    for k in recent_s:
        combined[f'recent_{k}'] = recent_s[k]
        combined[f'season_{k}'] = season_s[k] if season_s else recent_s[k]
    return combined


def get_h2h_stats(home_team, away_team, date, df, n=10):
    """Historial directo entre dos equipos."""
    h2h = df[
        ((df['HomeTeam'] == home_team) & (df['AwayTeam'] == away_team)) |
        ((df['HomeTeam'] == away_team) & (df['AwayTeam'] == home_team))
    ]
    h2h = h2h[h2h['Date'] < date].tail(n)

    if len(h2h) < 3:
        return {'h2h_home_wr': 0.33, 'h2h_draw_rate': 0.33, 'h2h_away_wr': 0.33}

    home_wins = ((h2h['HomeTeam'] == home_team) & (h2h['FTR'] == 'H')).sum() + \
                ((h2h['AwayTeam'] == home_team) & (h2h['FTR'] == 'A')).sum()
    draws     = (h2h['FTR'] == 'D').sum()
    away_wins = len(h2h) - home_wins - draws
    m = len(h2h)
    return {
        'h2h_home_wr':   home_wins / m,
        'h2h_draw_rate': draws / m,
        'h2h_away_wr':   away_wins / m,
    }


def get_tabla_posicion(team, date, df):
    """Posición y puntos en la tabla al momento del partido."""
    season = df[df['Date'] < date]['season'].iloc[-1]
    partidos_szn = df[(df['season'] == season) & (df['Date'] < date)]
    equipos = pd.concat([partidos_szn['HomeTeam'], partidos_szn['AwayTeam']]).unique()

    tabla = []
    for eq in equipos:
        home_p = partidos_szn[partidos_szn['HomeTeam'] == eq]
        away_p = partidos_szn[partidos_szn['AwayTeam'] == eq]
        pts = ((home_p['FTR']=='H').sum()*3 + (home_p['FTR']=='D').sum() +
               (away_p['FTR']=='A').sum()*3 + (away_p['FTR']=='D').sum())
        gf  = home_p['FTHG'].sum() + away_p['FTAG'].sum()
        gc  = home_p['FTAG'].sum() + away_p['FTHG'].sum()
        pj  = len(home_p) + len(away_p)
        tabla.append({'equipo': eq, 'pts': pts, 'gf': gf, 'gc': gc, 'pj': pj})

    tabla_df = pd.DataFrame(tabla).sort_values('pts', ascending=False).reset_index(drop=True)
    tabla_df['posicion']      = tabla_df.index + 1
    tabla_df['total_equipos'] = len(tabla_df)

    fila = tabla_df[tabla_df['equipo'] == team]
    if len(fila) == 0:
        return {'posicion': 10, 'pts_pg': 1.0, 'dif_goles_szn': 0, 'pct_posicion': 0.5}

    fila = fila.iloc[0]
    pj   = max(fila['pj'], 1)
    return {
        'posicion':      fila['posicion'],
        'pts_pg':        fila['pts'] / pj,
        'dif_goles_szn': (fila['gf'] - fila['gc']) / pj,
        'pct_posicion':  1 - (fila['posicion'] / fila['total_equipos']),
    }


def predecir_partido_v3(home_team, away_team, fecha_str):
    """Predice probabilidades H/D/A para un partido. Solo recomienda si confianza >55%."""
    fecha = pd.Timestamp(fecha_str)

    home_stats = get_team_stats_v2(home_team, fecha, df)
    away_stats = get_team_stats_v2(away_team, fecha, df)
    h2h        = get_h2h_stats(home_team, away_team, fecha, df)
    home_tabla = get_tabla_posicion(home_team, fecha, df)
    away_tabla = get_tabla_posicion(away_team, fecha, df)

    if home_stats is None or away_stats is None:
        print(f'Sin suficiente historia para {home_team} o {away_team}')
        return None

    row = {}
    for k, v in home_stats.items(): row[f'h_{k}'] = v
    for k, v in away_stats.items(): row[f'a_{k}'] = v
    for k, v in h2h.items():        row[k] = v
    for k, v in home_tabla.items(): row[f'h_tabla_{k}'] = v
    for k, v in away_tabla.items(): row[f'a_tabla_{k}'] = v

    row['dif_recent_wr']  = home_stats['recent_win_rate']  - away_stats['recent_win_rate']
    row['dif_season_wr']  = home_stats['season_win_rate']  - away_stats['season_win_rate']
    row['dif_recent_gf']  = home_stats['recent_gf_pg']     - away_stats['recent_gf_pg']
    row['dif_recent_gc']  = home_stats['recent_gc_pg']     - away_stats['recent_gc_pg']
    row['dif_recent_dif'] = home_stats['recent_dif_goles'] - away_stats['recent_dif_goles']
    row['dif_season_dif'] = home_stats['season_dif_goles'] - away_stats['season_dif_goles']
    row['home_advantage'] = home_stats['recent_home_wr']   - away_stats['recent_win_rate']
    row['dif_pts_pg']     = home_tabla['pts_pg']           - away_tabla['pts_pg']
    row['dif_posicion']   = away_tabla['posicion']         - home_tabla['posicion']

    X_new  = pd.DataFrame([row])[feature_cols_v3]
    proba  = model_v3.predict_proba(X_new)[0]

    prob_A, prob_D, prob_H = proba[0], proba[1], proba[2]
    confianza = max(proba)
    pred = le.inverse_transform([np.argmax(proba)])[0]
    usar = confianza >= 0.55

    resultado_label = {
        'H': f'{home_team} gana',
        'D': 'Empate',
        'A': f'{away_team} gana'
    }

    print(f"\n{'='*50}")
    print(f"  {home_team} vs {away_team}")
    print(f"{'='*50}")
    print(f"  Local gana     {home_team:<24} {prob_H:.1%}")
    print(f"  Empate                                    {prob_D:.1%}")
    print(f"  Visitante gana {away_team:<24} {prob_A:.1%}")
    print(f"{'─'*50}")
    print(f"  Prediccion:  {resultado_label[pred]}")
    print(f"  Confianza:   {confianza:.1%}  {'← USAR' if usar else '← IGNORAR'}")
    print(f"{'='*50}")

    return {
        'pred': pred, 'prob_H': prob_H, 'prob_D': prob_D,
        'prob_A': prob_A, 'confianza': confianza, 'usar': usar
    }


print('Funciones cargadas OK')

Funciones cargadas OK


In [31]:
# ─────────────────────────────────────────────
# CELDA 3 — Funciones de seguimiento
# Correr siempre al abrir el notebook
# ─────────────────────────────────────────────
predicciones = []

def registrar_prediccion(jornada, fecha, home, away, pred, prob_H, prob_D, prob_A, confianza):
    """Registra una predicción ANTES del partido."""
    predicciones.append({
        'jornada':    jornada,
        'fecha':      fecha,
        'home':       home,
        'away':       away,
        'prediccion': pred,
        'prob_H':     round(prob_H, 4),
        'prob_D':     round(prob_D, 4),
        'prob_A':     round(prob_A, 4),
        'confianza':  round(confianza, 4),
        'resultado':  None,
        'correcto':   None
    })
    print(f'Registrado: {home} vs {away} → {pred} ({confianza:.1%})')


def registrar_resultado(home, away, resultado_real):
    """
    Registra el resultado DESPUÉS del partido.
    resultado_real: 'H' (local gana), 'D' (empate), 'A' (visitante gana)
    """
    for p in predicciones:
        if p['home'] == home and p['away'] == away:
            p['resultado'] = resultado_real
            p['correcto']  = p['prediccion'] == resultado_real
            estado = 'ACERTADO ✓' if p['correcto'] else 'FALLADO ✗'
            print(f'{estado} — {home} vs {away} | Pred: {p["prediccion"]} | Real: {resultado_real}')
            return
    print(f'Partido no encontrado: {home} vs {away}')


def ver_seguimiento():
    """Muestra resumen completo de predicciones y accuracy acumulado."""
    if not predicciones:
        print('Sin predicciones registradas aún')
        return

    df_seg    = pd.DataFrame(predicciones)
    total     = len(df_seg)
    jugados   = df_seg['resultado'].notna().sum()
    acertados = df_seg['correcto'].sum() if jugados > 0 else 0
    accuracy  = acertados / jugados if jugados > 0 else 0

    print(f"\n{'='*52}")
    print(f"  RESUMEN GENERAL")
    print(f"{'='*52}")
    print(f"  Total predicciones:    {total}")
    print(f"  Partidos jugados:      {jugados}")
    print(f"  Acertados:             {int(acertados)}")
    print(f"  Accuracy real:         {accuracy:.1%}")
    print(f"  Baseline (siempre H):  41.2%")
    print(f"  Diferencia:            {(accuracy - 0.412):+.1%}")

    print(f"\n{'='*52}")
    print(f"  DETALLE POR PARTIDO")
    print(f"{'='*52}")
    for _, row in df_seg.iterrows():
        if row['resultado'] is not None:
            icon = '✓' if row['correcto'] else '✗'
            print(f"  {icon} J{row['jornada']} {row['home']} vs {row['away']}")
            print(f"    Pred: {row['prediccion']} ({row['confianza']:.1%}) | Real: {row['resultado']}")
        else:
            print(f"  ? J{row['jornada']} {row['home']} vs {row['away']}")
            print(f"    Pred: {row['prediccion']} ({row['confianza']:.1%}) | Pendiente")

    if jugados > 0:
        print(f"\n{'='*52}")
        print(f"  ACCURACY POR CONFIANZA")
        print(f"{'='*52}")
        df_jug = df_seg[df_seg['resultado'].notna()].copy()
        bins   = [0.50, 0.55, 0.60, 0.65, 1.0]
        labels = ['50-55%', '55-60%', '60-65%', '>65%']
        df_jug['conf_bin'] = pd.cut(df_jug['confianza'], bins=bins, labels=labels)
        resumen = df_jug.groupby('conf_bin', observed=True).agg(
            partidos=('correcto', 'count'),
            acertados=('correcto', 'sum'),
            accuracy=('correcto', 'mean')
        ).round(3)
        print(resumen.to_string())


def guardar_predicciones(archivo='predicciones.json'):
    """Guarda predicciones en disco para no perderlas al reiniciar el kernel."""
    datos = []
    for p in predicciones:
        p_copy = p.copy()
        p_copy['correcto'] = bool(p['correcto']) if p['correcto'] is not None else None
        datos.append(p_copy)
    with open(archivo, 'w') as f:
        json.dump(datos, f, indent=2)
    print(f'Guardado: {len(datos)} predicciones en {archivo}')


def cargar_predicciones(archivo='predicciones.json'):
    """Carga predicciones desde disco al reiniciar el kernel."""
    global predicciones
    try:
        with open(archivo, 'r') as f:
            predicciones = json.load(f)
        print(f'Cargado: {len(predicciones)} predicciones desde {archivo}')
    except FileNotFoundError:
        print('No hay predicciones guardadas aún — lista vacía')


print('Funciones de seguimiento cargadas OK')

Funciones de seguimiento cargadas OK


In [32]:
# ─────────────────────────────────────────────
# CELDA 4 — Predecir jornada
# Editar: partidos_jornada y fecha_jornada
# ─────────────────────────────────────────────

# ── EDITAR AQUÍ ──────────────────────────────
partidos_jornada = [
    ('West Ham',        'Wolves'),
    ('Arsenal',         'Bournemouth'),
    ('Brentford',       'Everton'),
    ('Burnley',         'Brighton'),
    ('Liverpool',       'Fulham'),
    ('Sunderland',      'Tottenham'),
    ("Nott'm Forest",   'Aston Villa'),
    ('Crystal Palace',  'Newcastle'),
    ('Chelsea',         'Man City'),
    ('Man United',      'Leeds'),
]

fecha_jornada = '2026-04-10'
# ─────────────────────────────────────────────

resultados_prediccion = {}
for home, away in partidos_jornada:
    res = predecir_partido_v3(home, away, fecha_jornada)
    if res:
        resultados_prediccion[f'{home} vs {away}'] = res


  West Ham vs Wolves
  Local gana     West Ham                 55.2%
  Empate                                    23.2%
  Visitante gana Wolves                   21.6%
──────────────────────────────────────────────────
  Prediccion:  West Ham gana
  Confianza:   55.2%  ← USAR

  Arsenal vs Bournemouth
  Local gana     Arsenal                  64.8%
  Empate                                    20.4%
  Visitante gana Bournemouth              14.8%
──────────────────────────────────────────────────
  Prediccion:  Arsenal gana
  Confianza:   64.8%  ← USAR

  Brentford vs Everton
  Local gana     Brentford                47.8%
  Empate                                    22.6%
  Visitante gana Everton                  29.6%
──────────────────────────────────────────────────
  Prediccion:  Brentford gana
  Confianza:   47.8%  ← IGNORAR

  Burnley vs Brighton
  Local gana     Burnley                  33.2%
  Empate                                    27.5%
  Visitante gana Brighton              

In [33]:
# ─────────────────────────────────────────────
# CELDA 5 — Registrar predicciones a usar
# Solo registrar las que tienen ← USAR
# Editar: jornada, fecha, home, away y valores
# ─────────────────────────────────────────────

# Cargar predicciones previas desde disco
cargar_predicciones()

# ── EDITAR AQUÍ — agregar solo las de confianza >55% ──
registrar_prediccion(
    jornada=32, fecha='2026-04-10',
    home='West Ham', away='Wolves',
    pred='H', prob_H=0.552, prob_D=0.232, prob_A=0.216,
    confianza=0.552
)

registrar_prediccion(
    jornada=32, fecha='2026-04-11',
    home='Arsenal', away='Bournemouth',
    pred='H', prob_H=0.648, prob_D=0.204, prob_A=0.148,
    confianza=0.648
)
# ──────────────────────────────────────────────────────

guardar_predicciones()
print('\nEstado actual:')
ver_seguimiento()

No hay predicciones guardadas aún — lista vacía
Registrado: West Ham vs Wolves → H (55.2%)
Registrado: Arsenal vs Bournemouth → H (64.8%)
Guardado: 2 predicciones en predicciones.json

Estado actual:

  RESUMEN GENERAL
  Total predicciones:    2
  Partidos jugados:      0
  Acertados:             0
  Accuracy real:         0.0%
  Baseline (siempre H):  41.2%
  Diferencia:            -41.2%

  DETALLE POR PARTIDO
  ? J32 West Ham vs Wolves
    Pred: H (55.2%) | Pendiente
  ? J32 Arsenal vs Bournemouth
    Pred: H (64.8%) | Pendiente


In [34]:
# # ─────────────────────────────────────────────
# # CELDA 6 — Registrar resultados reales
# # Correr DESPUÉS de cada partido
# #
# # H = gana el local
# # D = empate
# # A = gana el visitante
# # ─────────────────────────────────────────────

# # Cargar predicciones desde disco
# cargar_predicciones()

# # ── EDITAR AQUÍ — cambiar por resultado real ──
# registrar_resultado('West Ham',  'Wolves',      'H')  # ← cambiar después del partido
# registrar_resultado('Arsenal',   'Bournemouth', 'H')  # ← cambiar después del partido
# # ──────────────────────────────────────────────

# guardar_predicciones()

In [35]:
# ─────────────────────────────────────────────
# CELDA 7 — Ver resumen completo
# Correr en cualquier momento para ver el estado
# ─────────────────────────────────────────────

cargar_predicciones()
ver_seguimiento()

Cargado: 2 predicciones desde predicciones.json

  RESUMEN GENERAL
  Total predicciones:    2
  Partidos jugados:      0
  Acertados:             0
  Accuracy real:         0.0%
  Baseline (siempre H):  41.2%
  Diferencia:            -41.2%

  DETALLE POR PARTIDO
  ? J32 West Ham vs Wolves
    Pred: H (55.2%) | Pendiente
  ? J32 Arsenal vs Bournemouth
    Pred: H (64.8%) | Pendiente
